# Módulo 40 — Exercício: SVM

Neste exercício será utilizada a base **CARRO_CLIENTES.csv**, com o objetivo de comparar três modelos de classificação:

- SVM com kernel **linear**
- SVM com kernel **polinomial**
- XGBoost, reproduzindo a configuração utilizada no exercício anterior

A comparação será feita usando o mesmo conjunto de treino e teste para os três modelos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

import xgboost as xgb

pd.set_option("display.max_columns", None)

## 1. Carregamento e preparação inicial

Nesta etapa são verificados os tipos de dados, valores faltantes e a coluna de identificação é removida.

In [ ]:
base = pd.read_csv("CARRO_CLIENTES.csv")

print("Dimensões da base:", base.shape)
display(base.head())

print("\nTipos de dados:")
display(base.dtypes.to_frame("Tipo"))

print("\nDados faltantes:")
display(base.isnull().sum().to_frame("Valores faltantes"))

# Remove a coluna de identificação, pois ela não representa uma característica preditiva.
base = base.drop(columns=["User ID"])

print("\nColunas após remover User ID:")
print(base.columns.tolist())

## 2. Codificação das variáveis categóricas

A variável `Gender` é transformada em uma representação numérica com `LabelEncoder`. Em seguida, a coluna categórica original é removida.

In [ ]:
label_encoder = LabelEncoder()

base["Gender_encoded"] = label_encoder.fit_transform(base["Gender"])
mapeamento_gender = dict(
    zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))
)

base = base.drop(columns=["Gender"])

print("Mapeamento do Gender:", mapeamento_gender)
display(base.head())

## 3. Matriz de correlação

A correlação permite observar a associação linear entre cada variável numérica e o target `Purchased`.

In [ ]:
correlacao = base.corr(numeric_only=True)

plt.figure(figsize=(7, 5))
sns.heatmap(correlacao, annot=True, fmt=".3f")
plt.title("Matriz de correlação")
plt.tight_layout()
plt.show()

corr_target = (
    correlacao["Purchased"]
    .drop("Purchased")
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

print("Correlação com Purchased:")
display(corr_target.to_frame("Correlação"))

### Análise da correlação

Na base utilizada:

- **Age** apresenta a maior correlação com `Purchased`, com aproximadamente **0,616**.
- **AnnualSalary** aparece em seguida, com aproximadamente **0,365**.
- **Gender_encoded** possui correlação muito baixa, aproximadamente **-0,047**.

Assim, idade e salário anual parecem ser as variáveis mais relacionadas à propensão de compra do carro.

## 4. Separação em treino e teste

Os dados são divididos em 80% para treino e 20% para teste. O mesmo `random_state` e o mesmo split serão utilizados para os três modelos, garantindo uma comparação justa.

Como o SVM é sensível à escala das variáveis, `Age` e `AnnualSalary` são padronizadas com `StandardScaler`. O scaler é ajustado somente sobre o conjunto de treino, evitando vazamento de informação.

In [ ]:
X = base.drop(columns=["Purchased"])
Y = base["Purchased"]

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.20,
    random_state=42,
    stratify=Y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("Y_train:", Y_train.shape)
print("Y_test :", Y_test.shape)

print("\nDistribuição do target no teste:")
display(Y_test.value_counts().sort_index().to_frame("Quantidade"))

## 5. SVM com kernel linear

O primeiro modelo utiliza `kernel="linear"`. As métricas são calculadas no conjunto de **teste**, que representa dados não utilizados durante o treinamento.

In [ ]:
svm_linear = SVC(
    kernel="linear",
    C=1.0,
    probability=True,
    random_state=1
)

svm_linear.fit(X_train_scaled, Y_train)

pred_linear = svm_linear.predict(X_test_scaled)
prob_linear = svm_linear.predict_proba(X_test_scaled)[:, 1]

acc_linear = accuracy_score(Y_test, pred_linear)
prec_linear = precision_score(Y_test, pred_linear)
rec_linear = recall_score(Y_test, pred_linear)
f1_linear = f1_score(Y_test, pred_linear)
auc_linear = roc_auc_score(Y_test, prob_linear)

print("SVM LINEAR")
print(f"Acurácia : {acc_linear:.4f}")
print(f"Precisão : {prec_linear:.4f}")
print(f"Recall   : {rec_linear:.4f}")
print(f"F1-score : {f1_linear:.4f}")
print(f"ROC-AUC  : {auc_linear:.4f}")

print("\nRelatório de classificação:")
print(classification_report(Y_test, pred_linear, digits=4))

cm_linear = confusion_matrix(Y_test, pred_linear)

plt.figure(figsize=(5, 4))
sns.heatmap(cm_linear, annot=True, fmt="d", cbar=False)
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.title("Matriz de confusão — SVM Linear")
plt.tight_layout()
plt.show()

## 6. SVM com kernel polinomial

Agora é treinado um segundo SVM utilizando `kernel="poly"`, mantendo os mesmos conjuntos de treino e teste.

In [ ]:
svm_poly = SVC(
    kernel="poly",
    C=1.0,
    probability=True,
    random_state=1
)

svm_poly.fit(X_train_scaled, Y_train)

pred_poly = svm_poly.predict(X_test_scaled)
prob_poly = svm_poly.predict_proba(X_test_scaled)[:, 1]

acc_poly = accuracy_score(Y_test, pred_poly)
prec_poly = precision_score(Y_test, pred_poly)
rec_poly = recall_score(Y_test, pred_poly)
f1_poly = f1_score(Y_test, pred_poly)
auc_poly = roc_auc_score(Y_test, prob_poly)

print("SVM POLINOMIAL")
print(f"Acurácia : {acc_poly:.4f}")
print(f"Precisão : {prec_poly:.4f}")
print(f"Recall   : {rec_poly:.4f}")
print(f"F1-score : {f1_poly:.4f}")
print(f"ROC-AUC  : {auc_poly:.4f}")

print("\nRelatório de classificação:")
print(classification_report(Y_test, pred_poly, digits=4))

cm_poly = confusion_matrix(Y_test, pred_poly)

plt.figure(figsize=(5, 4))
sns.heatmap(cm_poly, annot=True, fmt="d", cbar=False)
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.title("Matriz de confusão — SVM Polinomial")
plt.tight_layout()
plt.show()

## 7. Comparação com o XGBoost do exercício anterior

Para tornar a comparação diretamente reproduzível neste notebook, o XGBoost é treinado com o **mesmo split de treino/teste** e com a mesma configuração utilizada no exercício anterior:

- `n_estimators=150`
- `max_depth=3`
- `learning_rate=0.05`
- `subsample=0.90`
- `colsample_bytree=0.90`

O XGBoost utiliza os valores originais das variáveis, pois modelos baseados em árvores não exigem padronização para funcionar adequadamente.

In [ ]:
modelo_xgb = xgb.XGBClassifier(
    objective="binary:logistic",
    n_estimators=150,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.90,
    colsample_bytree=0.90,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

modelo_xgb.fit(X_train, Y_train)

pred_xgb = modelo_xgb.predict(X_test)
prob_xgb = modelo_xgb.predict_proba(X_test)[:, 1]

acc_xgb = accuracy_score(Y_test, pred_xgb)
prec_xgb = precision_score(Y_test, pred_xgb)
rec_xgb = recall_score(Y_test, pred_xgb)
f1_xgb = f1_score(Y_test, pred_xgb)
auc_xgb = roc_auc_score(Y_test, prob_xgb)

print("XGBOOST")
print(f"Acurácia : {acc_xgb:.4f}")
print(f"Precisão : {prec_xgb:.4f}")
print(f"Recall   : {rec_xgb:.4f}")
print(f"F1-score : {f1_xgb:.4f}")
print(f"ROC-AUC  : {auc_xgb:.4f}")

print("\nRelatório de classificação:")
print(classification_report(Y_test, pred_xgb, digits=4))

In [ ]:
comparacao = pd.DataFrame({
    "Modelo": ["SVM Linear", "SVM Polinomial", "XGBoost"],
    "Acurácia": [acc_linear, acc_poly, acc_xgb],
    "Precisão": [prec_linear, prec_poly, prec_xgb],
    "Recall": [rec_linear, rec_poly, rec_xgb],
    "F1-score": [f1_linear, f1_poly, f1_xgb],
    "ROC-AUC": [auc_linear, auc_poly, auc_xgb]
})

comparacao = comparacao.sort_values("Acurácia", ascending=False).reset_index(drop=True)

print("Comparação dos modelos:")
display(comparacao.style.format({
    "Acurácia": "{:.4f}",
    "Precisão": "{:.4f}",
    "Recall": "{:.4f}",
    "F1-score": "{:.4f}",
    "ROC-AUC": "{:.4f}"
}))

comparacao_plot = comparacao.set_index("Modelo")[["Acurácia", "F1-score", "ROC-AUC"]]
comparacao_plot.plot(kind="bar", figsize=(9, 5))
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Comparação entre os modelos")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Análise comparativa final

Os três modelos foram avaliados sobre os **mesmos 200 registros de teste**, permitindo uma comparação direta.

### Resultados obtidos

| Modelo | Acurácia | Precisão | Recall | F1-score | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| **XGBoost** | **0,9150** | 0,8889 | **0,9000** | **0,8944** | **0,9666** |
| SVM Polinomial | 0,8750 | **0,9104** | 0,7625 | 0,8299 | 0,9310 |
| SVM Linear | 0,8550 | 0,8592 | 0,7625 | 0,8079 | 0,9239 |

O **XGBoost apresentou o melhor desempenho geral**. Ele obteve a maior acurácia (**91,5%**), o maior recall (**90,0%**), o maior F1-score (**0,8944**) e também o maior ROC-AUC (**0,9666**). Isso indica que, além de acertar uma proporção maior dos casos, ele conseguiu identificar melhor os clientes que realmente compraram o carro e apresentou maior capacidade de separar as duas classes.

Entre os modelos SVM, o **kernel polinomial foi superior ao linear** em acurácia, F1-score e ROC-AUC. A precisão do SVM polinomial foi a maior entre os três modelos, mas seu recall foi de apenas **76,25%**, o que significa que ele deixou de identificar uma quantidade maior de compradores reais quando comparado ao XGBoost.

Esse resultado também é coerente com a natureza dos algoritmos. O SVM linear procura uma fronteira de decisão linear, enquanto o kernel polinomial consegue representar relações mais complexas. Já o XGBoost combina várias árvores de decisão e consegue modelar de forma eficiente **relações não lineares e interações entre variáveis**, o que parece se adequar melhor ao padrão presente nesta base.

Portanto, considerando o conjunto das métricas e não apenas um único indicador, **o XGBoost foi o modelo com melhor desempenho nesta comparação**.